# KionTTS Production Training Pipeline (Multi-GPU / Dual T4 / A100 / Colab / Kaggle)
This production notebook implements the end-to-end training and standalone verification workflow for **KionTTS**:
- **Stage 1 (Acoustic Foundation & Style Distillation)**: Adapts PL-BERT, Duration Predictor, F0/Energy Predictor, and grounds `KionTagStyleEncoder` (Student) to the official pretrained StyleTTS2 voice manifold (Teacher).
- **Standalone Style Manifold Evaluation**: Dedicated cell to audit mathematical cosine similarity and L2 norms (Student vs Official LibriTTS Teacher) using `eval_checkpoint_alignment.py` on any saved checkpoint.
- **Stage 2 (Full-Stack HiFi-GAN Vocoder & Discriminators)**: End-to-end acoustic-to-waveform fine-tuning with multi-resolution STFT loss and MPD/MSD GAN guidance.
- **Continuous Remote Checkpoint Sync**: Uploads 500-step interval checkpoints and evaluation audio samples asynchronously to Hugging Face Hub.


## 1. System Setup & Dependencies
Installs required audio libraries (, , ), deep learning dependencies, and ensures monotonic alignment fallback availability.


In [ ]:
import os, sys, subprocess, torch

print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"    [GPU {i}] {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GB")

# 1. Install system audio libraries (espeak-ng is required for IPA phonemization)
!apt-get update -qq && apt-get install -y -qq espeak-ng libsndfile1 ffmpeg git git-lfs

# 2. Install deep learning and audio packages
!pip install -q cython phonemizer librosa soundfile torchaudio transformers munch pyworld praat-parselmouth huggingface_hub accelerate tensorboard einops einops-exts pydub pyyaml pyarrow fastparquet tqdm

# 3. Monotonic align (build safely; StyleTTS2 has pure-PyTorch fallback in utils.py)
!pip install -q git+https://github.com/resemble-ai/monotonic_align.git 2>/dev/null || true


## 2. Clone Repository & Setup Working Directory
Auto-detects runtime environment (Google Colab , Kaggle , or Local Workspace) and sets up the active directory and Python path.


In [ ]:
import os, sys

# Support Google Colab, Kaggle, and Local environments seamlessly
if os.path.exists("/kaggle"):
    WORKING_DIR = "/kaggle/working"
    REPO_DIR = os.path.join(WORKING_DIR, "KionTTS")
    if not os.path.exists(REPO_DIR):
        if os.path.exists("/kaggle/input/kiontts"):
            !cp -r /kaggle/input/kiontts /kaggle/working/KionTTS
        else:
            !git clone https://github.com/C1ph3r404/KionTTS.git /kaggle/working/KionTTS
elif os.path.exists("/content"):
    WORKING_DIR = "/content"
    REPO_DIR = os.path.join(WORKING_DIR, "KionTTS")
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/C1ph3r404/KionTTS.git /content/KionTTS
else:
    WORKING_DIR = os.getcwd()
    REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..")) if "__file__" in globals() else WORKING_DIR

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "StyleTTS2"))
os.chdir(REPO_DIR)
print(f"[✓] Active Repository Directory: {os.getcwd()}")


## 3. Download StyleTTS2 Pretrained Utilities
Downloads ASR aligner, JDC pitch extractor, PL-BERT phoneme embeddings, and the official StyleTTS2 LibriTTS 2nd stage checkpoint ().


In [ ]:
import os, urllib.request

def ensure_pretrained():
    base_dir = os.path.join(REPO_DIR, "StyleTTS2")
    asr_dir = os.path.join(base_dir, "Utils/ASR")
    jdc_dir = os.path.join(base_dir, "Utils/JDC")
    plb_dir = os.path.join(base_dir, "Utils/PLBERT")
    libri_dir = os.path.join(REPO_DIR, "Models/LibriTTS")

    os.makedirs(asr_dir, exist_ok=True)
    os.makedirs(jdc_dir, exist_ok=True)
    os.makedirs(plb_dir, exist_ok=True)
    os.makedirs(libri_dir, exist_ok=True)

    downloads = [
        ("ASR epoch_00080.pth", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/ASR/epoch_00080.pth", os.path.join(asr_dir, "epoch_00080.pth")),
        ("ASR config.yml", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/ASR/config.yml", os.path.join(asr_dir, "config.yml")),
        ("JDC bst.t7", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/JDC/bst.t7", os.path.join(jdc_dir, "bst.t7")),
        ("PLBERT step_1000000.t7", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/PLBERT/step_1000000.t7", os.path.join(plb_dir, "step_1000000.t7")),
        ("PLBERT config.yml", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/PLBERT/config.yml", os.path.join(plb_dir, "config.yml")),
    ]

    for name, url, dest in downloads:
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            print(f"[*] Downloading {name}...")
            urllib.request.urlretrieve(url, dest)
            print(f"[✓] Downloaded {name}")

    # Official LibriTTS 2nd stage pretrained backbone
    libri_pth = os.path.join(libri_dir, "epochs_2nd_00020.pth")
    if not os.path.exists(libri_pth) or os.path.getsize(libri_pth) < 100 * 1024 * 1024:
        print("[*] Downloading Pretrained LibriTTS 2nd stage checkpoint (epochs_2nd_00020.pth)...")
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(repo_id="yl4579/StyleTTS2-LibriTTS", filename="Models/LibriTTS/epochs_2nd_00020.pth", local_dir=REPO_DIR)
        print(f"[✓] LibriTTS Checkpoint ready: {downloaded}")

ensure_pretrained()


## 4. Hugging Face Hub Authentication & Checkpoint Sync Manager
Discovers API token across Kaggle Secrets, Colab Secrets (), environment variables, or HF cache, and initializes the asynchronous checkpoint synchronizer.


In [ ]:
import os
from kion_core.checkpoint_manager import KionCheckpointManager, get_hf_token

# 1. Retrieve Hugging Face token across platforms
hf_token = get_hf_token()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("[✓] Hugging Face token successfully authenticated!")
else:
    print("[!] Notice: HF_TOKEN not detected. Set 'HF_TOKEN' under Kaggle Secrets or Colab Secrets to enable remote sync.")

# 2. Checkpoint Manager initialized with HF repository
HF_REPO_ID = os.environ.get("HF_REPO_ID", "nate0001/TTS_test")
os.environ["HF_REPO_ID"] = HF_REPO_ID
CHECKPOINT_DIR = os.path.join(REPO_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

hf_manager = KionCheckpointManager(
    checkpoint_dir=CHECKPOINT_DIR,
    repo_id=HF_REPO_ID,
    hf_token=hf_token,
)


## 5. Dataset Extraction & Manifest Generation
Discovers local dataset formats (extracted batch folders, batch zips, tar archives, or sample dataset) or downloads and decodes Parquet shards from Hugging Face Hub.


In [ ]:
import os, sys, glob, json, subprocess

def find_dataset_source():
    out_dir = os.path.abspath(os.path.join(REPO_DIR, "DataSet"))
    out_manifest = os.path.join(out_dir, "train_manifest.json")

    search_roots = [
        "/kaggle/input",
        "/content/dataset",
        os.path.join(REPO_DIR, "DataSet/data"),
        os.path.join(REPO_DIR, "DataSet"),
        REPO_DIR,
    ]
    search_roots = [r for r in search_roots if os.path.exists(r)]

    def climb_to_main_root(sub_path):
        curr = os.path.dirname(os.path.abspath(sub_path)) if os.path.isfile(sub_path) else os.path.abspath(sub_path)
        while curr and curr not in ["/kaggle/input", "/kaggle", "/", os.path.expanduser("~")] and os.path.dirname(curr) != curr:
            parent = os.path.dirname(curr)
            base = os.path.basename(curr).lower()
            if any(marker in base for marker in ["batch", "tier", "kiontts_dataset_"]):
                curr = parent
            else:
                break
        return curr

    print("[*] Scanning for KionTTS dataset across search locations...")

    for root in search_roots:
        is_kaggle = root.startswith("/kaggle/input")
        prefix = "[Kaggle Input]" if is_kaggle else "[Local Workspace]"

        # 1. Sample Dataset format: train_manifest.json + wavs
        manifest_matches = glob.glob(os.path.join(root, "**", "train_manifest.json"), recursive=True)
        for m in manifest_matches:
            m_abs = os.path.abspath(m)
            if m_abs == out_manifest or os.path.dirname(m_abs) == out_dir:
                continue
            if os.path.getsize(m_abs) > 10:
                sample_root = os.path.dirname(m)
                print(f"[✓] {prefix} Found Dataset at: {sample_root}")
                return sample_root

        # 2. Main Dataset format (Extracted folders with batch metadata.json)
        meta_matches = glob.glob(os.path.join(root, "**", "metadata.json"), recursive=True)
        valid_metas = [
            m for m in meta_matches
            if any(k in m.lower() for k in ["batch", "tier", "kion", "tts"])
            and not os.path.abspath(m).startswith(out_dir)
        ]
        if valid_metas:
            main_root = climb_to_main_root(valid_metas[0])
            print(f"[✓] {prefix} Found Main Dataset extracted batches. Root: {main_root}")
            return main_root

        # 3. Main Dataset format (Batch zips)
        batch_zips = glob.glob(os.path.join(root, "**", "batch_*.zip"), recursive=True)
        if batch_zips:
            main_root = climb_to_main_root(batch_zips[0])
            print(f"[✓] {prefix} Found Main Dataset batch zips. Root: {main_root}")
            return main_root

        # 4. Main Dataset format (Tar archive)
        for pat in ["kion_dataset.tar", "*.tar", "*.tar.gz", "*.tgz"]:
            for t in glob.glob(os.path.join(root, "**", pat), recursive=True):
                if os.path.getsize(t) > 1024:
                    t_lower = os.path.basename(t).lower()
                    if any(k in t_lower for k in ["kion", "tts", "dataset"]):
                        print(f"[✓] {prefix} Found Dataset tar archive at: {t}")
                        return t

    return None

def is_valid_manifest(manifest_path):
    if not os.path.exists(manifest_path) or os.path.getsize(manifest_path) < 10:
        return False
    try:
        with open(manifest_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            return isinstance(data, list) and len(data) > 0
    except Exception:
        return False

manifest_train = os.path.join(REPO_DIR, "DataSet/train_manifest.json")
manifest_val = os.path.join(REPO_DIR, "DataSet/val_manifest.json")
dataset_out_dir = os.path.join(REPO_DIR, "DataSet")

if not is_valid_manifest(manifest_train):
    dataset_source = find_dataset_source()
    if dataset_source:
        print(f"[*] Preparing dataset from: {dataset_source} -> {dataset_out_dir}")
        !python3 scripts/prepare_dataset.py --source "{dataset_source}" --output_dir "{dataset_out_dir}"
    else:
        # Fallback: Download and decode Parquet shards from Hugging Face Hub
        print("[*] No local archive detected. Downloading dataset from Hugging Face: nate0001/TTS_test ...")
        from huggingface_hub import snapshot_download
        import pyarrow.parquet as pq
        import io, soundfile as sf

        snap_dir = snapshot_download(repo_id="nate0001/TTS_test", repo_type="dataset", token=hf_token, ignore_patterns=["*.git*", ".gitattributes", "README.md"])
        wavs_dir = os.path.join(dataset_out_dir, "wavs")
        os.makedirs(wavs_dir, exist_ok=True)

        parquet_files = sorted(glob.glob(os.path.join(snap_dir, "**", "*.parquet"), recursive=True))
        records = []
        for pfile in parquet_files:
            table = pq.read_table(pfile)
            for i in range(len(table)):
                sid = table["id"][i].as_py()
                txt = table["text"][i].as_py()
                audio_bytes = table["audio"][i].as_py()["bytes"]
                wpath = os.path.join(wavs_dir, f"{sid}.wav")
                if not os.path.exists(wpath):
                    with open(wpath, "wb") as wf:
                        wf.write(audio_bytes)
                records.append({"audio_filepath": os.path.relpath(wpath, dataset_out_dir), "text": txt})

        import random
        random.seed(42)
        random.shuffle(records)
        split = max(1, int(len(records) * 0.05))
        json.dump(records[split:], open(manifest_train, "w"), indent=2)
        json.dump(records[:split], open(manifest_val, "w"), indent=2)
        print(f"[✓] HF Parquet Dataset Decoded: {len(records)-split} train / {split} val samples.")
else:
    print(f"[✓] Valid train manifest already exists: {manifest_train}")

# Verify generated manifests
with open(manifest_train, "r", encoding="utf-8") as f:
    t_data = json.load(f)
v_data = json.load(open(manifest_val, "r", encoding="utf-8")) if os.path.exists(manifest_val) else []
print(f"[✓] Dataset Ready: {len(t_data)} train samples | {len(v_data)} validation samples.")


## 6. Stage 1 Training: Acoustic Foundation & Tag Style Alignment
Adapts PL-BERT, Duration Predictor, F0/Energy Predictor, and trains `KionTagStyleEncoder` (Student) against the official pretrained StyleTTS2 voice manifold (Teacher).


In [ ]:
import subprocess

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
launch_cmd = f"torchrun --nproc_per_node={num_gpus}" if num_gpus > 1 else "python3"
token_arg = f"--hf_token {hf_token}" if hf_token else ""

stage1_cmd = f"""{launch_cmd} scripts/train_stage1.py \
    --config StyleTTS2/Configs/config_ft.yml \
    --pretrained_ckpt Models/LibriTTS/epochs_2nd_00020.pth \
    --manifest DataSet/train_manifest.json \
    --val_manifest DataSet/val_manifest.json \
    --data_root DataSet \
    --checkpoint_dir checkpoints \
    --epochs 40 \
    --batch_size 4 \
    --accum_steps 2 \
    --save_step_freq 500 \
    --hf_repo {HF_REPO_ID} \
    {token_arg} \
    --lr 0.0001
"""

print(f"[*] Launching Stage 1 Training:\n{stage1_cmd}\n")
ret = subprocess.run(stage1_cmd, shell=True)
if ret.returncode != 0:
    raise RuntimeError(f"[FATAL] Stage 1 Training failed with exit code {ret.returncode}! Aborting pipeline before Stage 2.")
print("[✓] Stage 1 Training completed successfully!")


## 6.1 Standalone Checkpoint Style Manifold Evaluation
**Separate on-demand evaluation cell:** Audits any saved checkpoint against the validation set without disrupting training.
Validates that:
- **Teacher Vector Norm**: finite & healthy (`~4.61`)
- **Student Vector Norm**: bounded (`~1.63`)
- **Acoustic Cosine Sim (128D)**: `> +0.80 to +0.95`
- **Prosodic Cosine Sim (128D)**: `> +0.70 to +0.85`
- **Coordinate MSE Loss**: `< 0.05`


In [ ]:
# Standalone Checkpoint-Level Style Manifold Evaluator
# You can run this separate cell at any time on any saved checkpoint
import glob

# Set a specific checkpoint to evaluate (e.g. 'checkpoints/kion_stage1_epoch_000_step_000500.pth')
# Or leave as None to automatically audit the latest available checkpoint:
target_checkpoint = None

if target_checkpoint:
    !python3 scripts/eval_checkpoint_alignment.py "{target_checkpoint}"
else:
    !python3 scripts/eval_checkpoint_alignment.py


## 7. Stage 2 Training: Full-Stack HiFi-GAN Vocoder & GAN Discriminators
Joint acoustic-to-waveform fine-tuning with Multi-Resolution STFT reconstruction loss and MPD/MSD GAN guidance.


In [ ]:
import subprocess, glob

# 1. Safety check: ensure a valid Stage 1 checkpoint exists before launching Stage 2
stage1_ckpt = hf_manager.find_latest_checkpoint(stage="stage1")
if not stage1_ckpt or not os.path.exists(stage1_ckpt):
    local_ckpts = sorted(glob.glob("checkpoints/kion_stage1_*.pth") + glob.glob("checkpoints/*step*.pth"))
    if local_ckpts:
        stage1_ckpt = local_ckpts[-1]
    else:
        raise RuntimeError("[FATAL] Cannot start Stage 2: No valid Stage 1 checkpoint found! Stage 1 must finish successfully first.")
print(f"[✓] Verified Stage 1 checkpoint for Stage 2: {stage1_ckpt}")

# 2. Adaptive batch size: on 15-16GB GPUs (dual T4s), use batch_size 2 with accum_steps 4 to prevent HiFi-GAN VRAM OOM
gpu_mem_gb = (torch.cuda.get_device_properties(0).total_memory / (1024**3)) if torch.cuda.is_available() else 0
stage2_bs = 2 if (0 < gpu_mem_gb < 20) else 4
stage2_accum = 4 if stage2_bs == 2 else 2
print(f"[*] Stage 2 configuration: batch_size={stage2_bs}, accum_steps={stage2_accum} (VRAM: {gpu_mem_gb:.1f} GB)")

token_arg = f"--hf_token {hf_token}" if hf_token else ""
stage2_cmd = f"""{launch_cmd} scripts/train_stage2.py \n    --stage1_ckpt {stage1_ckpt} \n    --pretrained_ckpt Models/LibriTTS/epochs_2nd_00020.pth \n    --manifest DataSet/train_manifest.json \n    --val_manifest DataSet/val_manifest.json \n    --data_root DataSet \n    --checkpoint_dir checkpoints \n    --epochs 50 \n    --batch_size {stage2_bs} \n    --accum_steps {stage2_accum} \n    --save_step_freq 500 \n    --dec_window 48 \n    --hf_repo {HF_REPO_ID} \n    {token_arg} \n    --lr 0.0001
"""

print(f"[*] Launching Stage 2 Training:
{stage2_cmd}
")
ret = subprocess.run(stage2_cmd, shell=True)
if ret.returncode != 0:
    raise RuntimeError(f"[FATAL] Stage 2 Training failed with exit code {ret.returncode}! Aborting pipeline.")
print("[✓] Stage 2 Training completed successfully!")


## 8. Interactive Audio Inference & Emotional Tag Testing
Synthesizes custom speech prompts with direct inline emotional and delivery style tags.


In [ ]:
import IPython.display as ipd
import soundfile as sf
from kion_core.synthesizer import KionSynthesizer
from kion_core.tag_style_encoder import KionTagStyleEncoder
from scripts.train_stage1 import load_styletts2_backbone

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load backbone & tag encoder
model, _ = load_styletts2_backbone("StyleTTS2/Configs/config_ft.yml", "Models/LibriTTS/epochs_2nd_00020.pth", device=device)
tag_encoder = KionTagStyleEncoder(num_tags=24, emb_dim=64, style_dim=256)

# Find latest checkpoint
best_ckpt = hf_manager.find_latest_checkpoint(stage="stage2") or hf_manager.find_latest_checkpoint(stage="stage1")
if not best_ckpt or not os.path.exists(best_ckpt):
    local_ckpts = sorted(glob.glob("checkpoints/*.pth"))
    if local_ckpts:
        best_ckpt = local_ckpts[-1]
    else:
        raise RuntimeError("[FATAL] No checkpoints found to run inference! Pipeline execution incomplete.")

print(f"[*] Loading trained checkpoint: {best_ckpt}")
hf_manager.load_checkpoint(
    best_ckpt,
    models={
        "tag_encoder": tag_encoder,
        "predictor": model.get("predictor"),
        "decoder": model.get("decoder"),
        "bert": model.get("bert"),
        "bert_encoder": model.get("bert_encoder"),
        "text_encoder": model.get("text_encoder"),
    }
)

synthesizer = KionSynthesizer(model=model, tag_encoder=tag_encoder, device=device)

# Test prompts across different emotions
test_prompts = [
    "[happy=0.9, excited=0.8] We did it! The complete speech pipeline has converged with studio grade quality!",
    "[sarcasm=0.9] Oh, brilliant. Another batch of GPU tensor operations to keep me entertained.",
    "[soothing=0.85, calm=0.7] Take a deep breath. Every phoneme is aligning perfectly.",
    "[angry=0.85] I told you three times already, do not interrupt the training step!",
]

for i, p in enumerate(test_prompts):
    print(f"
Generating: {p}")
    wav = synthesizer.synthesize(prompt=p)
    out_path = f"kion_test_sample_{i+1}.wav"
    sf.write(out_path, wav, 24000)
    ipd.display(ipd.Audio(out_path, rate=24000))


## 9. Final Model Bundle Export & Release
Bundles trained KionTTS checkpoints, architecture configs, and phoneme dictionaries into a standalone production release directory.


In [ ]:
from kion_core.export import export_kion_model

release_dir = "kiontts_production_release"
export_kion_model(
    model=model,
    tag_encoder=tag_encoder,
    output_dir=release_dir,
)
print(f"[✓] Model successfully bundled for production deployment at: {release_dir}")
